In [1]:
import os

os.environ["POLARS_MAX_THREADS"] = "64"
import polars as pl

print("Polars threads:", pl.thread_pool_size())

import json
import scanpy as sc
import pandas as pd
import numpy as np

Polars threads: 64


In [2]:
# Load in the snrna anndata
snrna = sc.read_h5ad(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/converted_snRNA_data.h5ad"
)
snrna

/usr/local/lib/python3.12/dist-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 77604 × 36601
    obs: 'Sample', 'Barcode', 'key', 'SAMPLE_ID', 'pos', 'BrNum', 'round', 'Position', 'age', 'sex', 'diagnosis', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'high_mito', 'low_sum', 'low_detected', 'discard_auto', 'doubletScore', 'prelimCluster', 'collapsedCluster', 'kmeans', 'sizeFactor', 'cellType_broad_k', 'cellType_k', 'cellType_broad_hc', 'cellType_hc', 'cellType_layer', 'layer_annotation'
    var: 'source', 'type', 'gene_id', 'gene_version', 'gene_name', 'gene_type', 'binomial_deviance'
    uns: 'Samples', 'X_name', 'cell_type_colors', 'cell_type_colors_broad'
    obsm: 'GLMPCA_approx', 'HARMONY', 'TSNE', 'UMAP'
    layers: 'logcounts'

In [3]:
cell_meta = pl.from_pandas(snrna.obs.reset_index(names="cell_id")).with_columns(pl.col("cell_id").cast(pl.Categorical))
cell_meta

cell_id,Sample,Barcode,key,SAMPLE_ID,pos,BrNum,round,Position,age,sex,diagnosis,sum,detected,subsets_Mito_sum,subsets_Mito_detected,subsets_Mito_percent,total,high_mito,low_sum,low_detected,discard_auto,doubletScore,prelimCluster,collapsedCluster,kmeans,sizeFactor,cellType_broad_k,cellType_k,cellType_broad_hc,cellType_hc,cellType_layer,layer_annotation
cat,cat,cat,str,cat,cat,cat,cat,cat,f64,cat,cat,f64,i32,f64,i32,f64,f64,bool,bool,bool,bool,f64,cat,cat,cat,f64,cat,cat,cat,cat,cat,cat
"""1_AAACCCAAGTTCTCTT-1""","""Br2720_mid""","""AAACCCAAGTTCTCTT-1""","""AAACCCAAGTTCTCTT-1_Br2720_mid""","""1c-k""","""mid""","""Br2720""","""round1""","""Middle""",48.22,"""F""","""Control""",23075.0,5837,52.0,13,0.225352,23075.0,false,false,false,false,1.399882,"""76""","""HC04""","""mbk25""",1.689259,"""Inhib""","""Inhib_04""","""Inhib""","""Inhib_01""","""Inhib""","""L2/3"""
"""1_AAACCCACAAGGTCTT-1""","""Br2720_mid""","""AAACCCACAAGGTCTT-1""","""AAACCCACAAGGTCTT-1_Br2720_mid""","""1c-k""","""mid""","""Br2720""","""round1""","""Middle""",48.22,"""F""","""Control""",4046.0,2040,12.0,7,0.296589,4046.0,false,false,false,false,0.157206,"""18""","""HC07""","""mbk11""",0.296197,"""Oligo""","""Oligo_02""","""Oligo""","""Oligo_02""","""Oligo""","""WM"""
"""1_AAACCCATCAAAGACA-1""","""Br2720_mid""","""AAACCCATCAAAGACA-1""","""AAACCCATCAAAGACA-1_Br2720_mid""","""1c-k""","""mid""","""Br2720""","""round1""","""Middle""",48.22,"""F""","""Control""",7104.0,2858,23.0,10,0.323761,7104.0,false,false,false,false,0.209608,"""272""","""HC12""","""mbk07""",0.520065,"""OPC""","""OPC""","""OPC""","""OPC""","""OPC""","""L1/WM"""
"""1_AAACCCATCATGACAC-1""","""Br2720_mid""","""AAACCCATCATGACAC-1""","""AAACCCATCATGACAC-1_Br2720_mid""","""1c-k""","""mid""","""Br2720""","""round1""","""Middle""",48.22,"""F""","""Control""",28025.0,6272,25.0,9,0.089206,28025.0,false,false,false,false,0.381786,"""57""","""HC04""","""mbk25""",2.051635,"""Inhib""","""Inhib_04""","""Inhib""","""Inhib_01""","""Inhib""","""L2/3"""
"""1_AAACCCATCATTGCGA-1""","""Br2720_mid""","""AAACCCATCATTGCGA-1""","""AAACCCATCATTGCGA-1_Br2720_mid""","""1c-k""","""mid""","""Br2720""","""round1""","""Middle""",48.22,"""F""","""Control""",2903.0,1406,21.0,9,0.72339,2903.0,false,false,false,false,0.07486,"""18""","""HC07""","""mbk11""",0.212521,"""Oligo""","""Oligo_02""","""Oligo""","""Oligo_02""","""Oligo""","""WM"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""19_TTTGTTGGTAGGGAGG-1""","""Br2743_mid""","""TTTGTTGGTAGGGAGG-1""","""TTTGTTGGTAGGGAGG-1_Br2743_mid""","""round0""","""mid""","""Br2743""","""round0""","""Middle""",61.54,"""M""","""Control""",10089.0,4349,6.0,5,0.059471,10089.0,false,false,false,false,0.10278,"""41""","""HC02""","""mbk15""",0.738588,"""Excit""","""Excit_08""","""Excit""","""Excit_01""","""Excit_L3""","""L3"""
"""19_TTTGTTGGTCTTCATT-1""","""Br2743_mid""","""TTTGTTGGTCTTCATT-1""","""TTTGTTGGTCTTCATT-1_Br2743_mid""","""round0""","""mid""","""Br2743""","""round0""","""Middle""",61.54,"""M""","""Control""",32224.0,7119,3.0,3,0.00931,32224.0,false,false,false,false,1.945968,"""48""","""HC19""","""mbk25""",2.359032,"""Inhib""","""Inhib_04""","""Inhib""","""Inhib_05""","""Inhib""","""L2"""
"""19_TTTGTTGTCCCTGGTT-1""","""Br2743_mid""","""TTTGTTGTCCCTGGTT-1""","""TTTGTTGTCCCTGGTT-1_Br2743_mid""","""round0""","""mid""","""Br2743""","""round0""","""Middle""",61.54,"""M""","""Control""",42618.0,8329,4.0,3,0.009386,42618.0,false,false,false,false,1.06206,"""20""","""HC02""","""mbk03""",3.119949,"""Excit""","""Excit_02""","""Excit""","""Excit_01""","""Excit_L3""","""L3"""


In [4]:
cell_meta.write_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/cell_meta.parquet"
)

In [5]:
snrna.var_names = snrna.var["gene_id"]
gene_meta = pl.from_pandas(snrna.var).with_columns(
    pl.col("gene_id").cast(pl.Categorical)
)
print(gene_meta.select(pl.col("gene_name").n_unique()))
print(gene_meta.select(pl.col("gene_id").n_unique()))

gene_meta

shape: (1, 1)
┌───────────┐
│ gene_name │
│ ---       │
│ u32       │
╞═══════════╡
│ 36591     │
└───────────┘
shape: (1, 1)
┌─────────┐
│ gene_id │
│ ---     │
│ u32     │
╞═════════╡
│ 36601   │
└─────────┘


source,type,gene_id,gene_version,gene_name,gene_type,binomial_deviance
cat,cat,cat,cat,cat,cat,f64
"""HAVANA""","""gene""","""ENSG00000243485""","""5""","""MIR1302-2HG""","""lncRNA""",null
"""HAVANA""","""gene""","""ENSG00000237613""","""2""","""FAM138A""","""lncRNA""",null
"""HAVANA""","""gene""","""ENSG00000186092""","""6""","""OR4F5""","""protein_coding""",null
"""HAVANA""","""gene""","""ENSG00000238009""","""6""","""AL627309.1""","""lncRNA""",8778.743973
"""HAVANA""","""gene""","""ENSG00000239945""","""1""","""AL627309.3""","""lncRNA""",914.70276
…,…,…,…,…,…,…
"""ENSEMBL""","""gene""","""ENSG00000277836""","""1""","""AC141272.1""","""protein_coding""",null
"""ENSEMBL""","""gene""","""ENSG00000278633""","""1""","""AC023491.2""","""protein_coding""",null
"""ENSEMBL""","""gene""","""ENSG00000276017""","""1""","""AC007325.1""","""protein_coding""",null


In [6]:
gene_meta.write_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/gene_meta.parquet"
)

In [7]:
# cell_ids = snrna.obs_names.to_list()
# gene_ids = snrna.var_names.to_list()


# df_long = (
#     # Long raw counts
#     pl.DataFrame(snrna.X, schema=gene_ids)
#     .with_columns(pl.Series("cell_id", cell_ids).cast(pl.Categorical))
#     .unpivot(
#         index="cell_id",
#         variable_name="gene_id",
#         value_name="rawcount",
#     )
#     .with_columns(
#         [pl.col("rawcount").cast(pl.UInt32), pl.col("gene_id").cast(pl.Categorical)]
#     )
#     .join(
#         # Long logcounts
#         (
#             pl.DataFrame(snrna.layers["logcounts"], schema=gene_ids)
#             .with_columns(pl.Series("cell_id", cell_ids).cast(pl.Categorical))
#             .unpivot(
#                 index="cell_id",
#                 variable_name="gene_id",
#                 value_name="logcount",
#             )
#             .with_columns(
#                 pl.col("gene_id").cast(pl.Categorical),
#             )
#         ),
#         on=["cell_id", "gene_id"],
#         how="inner",
#     )
# )
# df_long

In [8]:
# df_long.write_parquet("/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/snrna_long.parquet")

In [9]:
# df_log

In [10]:
# raw_long = df_raw.unpivot(
#     index="cell_id",         # what was id_vars in melt
#     variable_name="gene_id", # column names become this
#     value_name="rawcount"    # column values become this
# )
# raw_long

In [11]:
# IDs as lists
cell_ids = snrna.obs_names.to_list()
gene_ids = snrna.var_names.to_list()
print(f"Number of cells: {len(cell_ids)}")
print(f"Number of genes: {len(gene_ids)}")

# Convert X and logcounts to numpy (if not already)
X_raw = np.asarray(snrna.X)
X_log = np.asarray(snrna.layers["logcounts"])
print("converted X to numpy arrays")

# Get the non-zero entries (sparse-ish approach)
cell_idx, gene_idx = np.nonzero(X_raw)
print("got non-zero indices")

# Map indices to IDs
cell_col = [cell_ids[i] for i in cell_idx]
gene_col = [gene_ids[i] for i in gene_idx]
print("mapped indices to IDs")

# Extract values
raw_vals = X_raw[cell_idx, gene_idx]
log_vals = X_log[cell_idx, gene_idx]
print("extracted values")

# Build Polars DataFrame directly in tall form
df_tall = pl.DataFrame(
    {
        "cell_id": cell_col,
        "gene_id": gene_col,
        "rawcount": raw_vals,
        "logcount": log_vals,
    }
).with_columns(
    [
        pl.col("cell_id").cast(pl.Categorical),
        pl.col("gene_id").cast(pl.Categorical),
        pl.col("rawcount").cast(pl.UInt32),
    ]
)

Number of cells: 77604
Number of genes: 36601
converted X to numpy arrays
got non-zero indices
mapped indices to IDs
extracted values


In [12]:
df_tall.write_parquet(
    "/zata/zippy/kresgeb/nmf_stuff/dlPFC/data/snrna/snrna_nonzero_counts_tall.parquet"
)

In [13]:
df_tall

cell_id,gene_id,rawcount,logcount
cat,cat,u32,f64
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000188976""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000188290""",2,1.126941
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000131591""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000078808""",1,0.670818
"""1_AAACCCAAGTTCTCTT-1""","""ENSG00000131584""",2,1.126941
…,…,…,…
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198804""",5,1.194889
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198712""",1,0.330965
"""19_TTTGTTGTCTTAAGGC-1""","""ENSG00000198938""",2,0.599993
